# OpenSubtaskTrain checkpoint rollout videos

Loads the most recent checkpoint for each seed launched by `open_subtask_train.slurm` in the sibling `maniskill_open_subtask_train/` folder (`#SBATCH --array=12-16`, i.e. seeds 12-16 under `LOG_DIR=logs/ppo_maniskill_open_subtask_train_adjacent_room/`) and renders `NUM_VIDEOS_PER_CHECKPOINT` rollout videos per checkpoint.

Sibling of `../maniskill_close_subtask_train/render_checkpoint_videos.ipynb` -- same scene, same adjacent-room spawn, same rollout/render helpers (`ppo_video_utils.build_networks` / `rollout_and_render` / `write_video`), only the goal is reversed (drawer starts closed, success means opening it -- see `env_utils.ManiskillOpenSubtaskTrain` / `contrastive.utils.DrawerOpenSuccessObserver`). Env-var setup mirrors `logs/spawn_check/base_standoff_explorer.ipynb`'s setup cell -- a Jupyter kernel never sees `open_subtask_train.slurm`'s `export` lines, so they're set here instead.

Run cells top to bottom once; re-run just the last ("View videos") cell to redisplay without re-rendering.

## Setup

In [ ]:
import os
import sys

REPO_ROOT = '/home/gliu2/dist_matching/sgcrl'
if REPO_ROOT not in sys.path:
  sys.path.insert(0, REPO_ROOT)

# These normally come from close_subtask_train.slurm's `export` lines -- a
# Jupyter kernel launched from the IDE never sees those, so they must be set
# here instead, BEFORE the heavy imports below:
#   - MS_ASSET_DIR/MSHAB_* -- else `_mshab_close_subtask_paths()` silently
#     falls back to mani_skill's default (empty) asset dir and fails with a
#     "Path ... not found" AssertionError.
#   - PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION -- else `import contrastive`
#     (which pulls in reverb, built against an older generated-proto ABI)
#     raises `TypeError: Descriptors cannot be created directly` against a
#     newer protobuf runtime (same tensorflow==2.8/reverb-vs-newer-protobuf
#     conflict close_subtask_train.slurm's comment describes).
#   - LD_LIBRARY_PATH/CPATH/XLA_PYTHON_CLIENT_MEM_FRACTION -- CUDA/mujoco
#     libs jax/torch/sapien dlopen later at first use; setting os.environ
#     here (before those libraries are touched) is early enough for that
#     lazy dlopen to pick them up, same as the MS_ASSET_DIR pattern above.
os.environ.setdefault('MS_ASSET_DIR', '/data/user_data/gliu2/maniskill_data')
os.environ.setdefault('MSHAB_TASK', 'set_table')
os.environ.setdefault('MSHAB_SPLIT', 'train')
os.environ.setdefault('MSHAB_OBJ', 'kitchen_counter')
os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '.4')
os.environ.setdefault('CPATH', '/data/user_data/gliu2/conda_envs/mshab_rl/include')

_ld_path_parts = [
    '/usr/local/cuda-11/lib64',
    '/data/user_data/gliu2/conda_envs/mshab_rl/lib',
    '/home/gliu2/.mujoco/mujoco210/bin',
    '/usr/lib/nvidia',
]
try:
  # Torch's own bundled (correct) libcusolver must come first -- same fix
  # as close_subtask_train.slurm's matching comment.
  import nvidia.cusolver as _cusolver
  _ld_path_parts.insert(
      0, os.path.join(os.path.dirname(_cusolver.__file__), 'lib'))
except ImportError:
  pass
os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(
    _ld_path_parts + [os.environ.get('LD_LIBRARY_PATH', '')]).rstrip(os.pathsep)

import sgcrl_jax_acme_compat  # noqa: F401 -- must precede all acme/jax imports

from IPython.display import Video, display

import ppo_video_utils
from contrastive import ppo_learner
from ppo_contrastive import fixed_goal_dict
from ppo_rollout_video import _enumerate_checkpoints

## Config

`HIDDEN_LAYER_SIZES`/`SEEDS`/`LOG_DIR` match `open_subtask_train.slurm`'s `--hidden_layer_sizes` flag, `#SBATCH --array=12-16`, and `LOG_DIR=` line respectively. `_enumerate_checkpoints` (from `ppo_rollout_video.py`) resolves the most recent checkpoint per run dir -- the highest `ckpt_iter_*.pkl` if present, else `latest.pkl`.

In [ ]:
ENV_NAME = 'maniskill_open_subtask_train'
HIDDEN_LAYER_SIZES = (256, 256, 256, 256, 256, 256)
SEEDS = [12]
LOG_DIR = '/home/gliu2/dist_matching/sgcrl/logs/ppo_maniskill_open_subtask_train_adjacent_room/'
NUM_VIDEOS_PER_CHECKPOINT = 10
# Training's own periodic eval loop (ppo_learner.py's `act_and_value` ->
# `networks.sample(dist, rng)`) always SAMPLES actions, never uses the
# deterministic mean -- confirmed empirically for the close-subtask sibling
# notebook (seed 4, latest.pkl, 8 rollouts each): stochastic=False -> 0/8
# drawer_closed successes, stochastic=True -> 3/8. Not yet re-measured for
# this open-subtask env specifically, but the training loop this policy was
# produced by is identical (same ppo_learner.py eval path) -- matching
# STOCHASTIC=False here keeps this notebook's success rate comparable to
# the logged eval CSV / success_1000 metric, same as the close notebook;
# revisit if this env's checkpoints turn out to behave differently.
STOCHASTIC = False
FPS = 30
OUTPUT_ROOT = os.path.join(LOG_DIR, 'rollout_videos_notebook')

checkpoint_paths = {}
for seed in SEEDS:
  ckpt_dir = os.path.join(
      LOG_DIR, f'ppo_maniskill_open_subtask_train_{seed}', 'checkpoints')
  entries = _enumerate_checkpoints(ckpt_dir)
  if not entries:
    raise FileNotFoundError(f'No checkpoints found under {ckpt_dir}')
  label, path = entries[-1]  # most recent: highest ckpt_iter_*, else latest.pkl
  checkpoint_paths[seed] = (label, path)
  print(f'seed={seed}: using checkpoint {label!r} ({path})')

## Build networks + env once

All three checkpoints share the same architecture/env config (verified against each run's `run_config.json`), so -- same reasoning as `ppo_rollout_video.py`'s "build ONCE and reuse" comment -- the env, render context, and jitted policy graph are built a single time and only `policy_params` changes per checkpoint below.

In [ ]:
print('[setup] building networks and env (shared across all checkpoints)...')
networks, _obs_dim, gym_env, env_max_steps = ppo_video_utils.build_networks(
    ENV_NAME, seed=0, hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    fixed_start_end=fixed_goal_dict[ENV_NAME], render_mode='rgb_array')
render_fn = ppo_video_utils.get_render_fn(ENV_NAME, gym_env)
print(f'[setup] env_max_steps={env_max_steps}')

## Success metric helper

Shared by both the "Render videos" and "Find successful rollouts" sections below. This env's `reward` is mshab's *strict* success (joint open AND arm at rest AND robot static); training's own eval logging (`contrastive.utils.DrawerOpenSuccessObserver`) instead reads the narrower `info['drawer_open']` (joint open only) -- use this `success_fn` with `rollout_and_render` to match that logged metric.

In [ ]:
def _drawer_open_success_fn(reward, done, info):
  del reward, done  # unused -- only the joint-open flag matters here
  return bool(info.get('drawer_open', False))

## Render videos

For each seed's checkpoint: load `policy_params` + `obs_normalizer_state` (checkpoints were trained with `ppo_norm_obs=True`, so the policy expects normalized observations), then roll out `NUM_VIDEOS_PER_CHECKPOINT` episodes. `OpenSubtaskTrain-v0`'s own `reset()` randomizes the scene/spawn each call (see `env_utils.ManiskillOpenSubtaskTrain.reset`), so repeated rollouts against the same policy naturally show different scenes/spawns without needing an explicit per-rollout env seed.

In [ ]:
video_paths = {seed: [] for seed in SEEDS}

# NOTE: `_drawer_open_success_fn` is defined in the "Success metric
# helper" cell above -- shared with "Find successful rollouts" below.
# `gym_env` is NOT closed here (unlike earlier versions of this cell) --
# it's shared with that later section too; see this notebook's final
# "Cleanup" cell for the actual `gym_env.close()`.
for seed in SEEDS:
  label, ckpt_path = checkpoint_paths[seed]
  print(f'\n=== seed={seed}  checkpoint={label} ===')
  ckpt = ppo_learner.load_checkpoint(ckpt_path)
  policy_params = ckpt['policy_params']
  print(f'  iteration={ckpt.get("iteration")}  global_step={ckpt.get("global_step")}')

  obs_norm_state = ckpt.get('obs_normalizer_state')
  obs_normalizer = (
      ppo_learner.ObsNormalizer.from_state_dict(obs_norm_state)
      if obs_norm_state is not None else None)
  if obs_normalizer is None:
    print('  WARNING: no obs_normalizer_state in checkpoint -- feeding raw obs.')

  out_dir = os.path.join(OUTPUT_ROOT, f'seed_{seed}')
  os.makedirs(out_dir, exist_ok=True)
  for i in range(NUM_VIDEOS_PER_CHECKPOINT):
    frames, stats = ppo_video_utils.rollout_and_render(
        policy_params=policy_params,
        gym_env=gym_env,
        networks=networks,
        render_fn=render_fn,
        max_steps=env_max_steps,
        stochastic=STOCHASTIC,
        seed=i,
        obs_normalizer=obs_normalizer,
        success_fn=_drawer_open_success_fn,
    )
    out_path = os.path.join(out_dir, f'rollout_{i}.mp4')
    ppo_video_utils.write_video(frames, out_path, fps=FPS)
    video_paths[seed].append(out_path)
    print(f'  [{i}] length={stats["length"]:4d}  total_reward(strict)={stats["total_reward"]:.3f}  '
          f'drawer_open_success={stats["success"]}  -> {out_path}')

print('\nDone.')

## View videos

Re-run this cell on its own (no need to re-render) to redisplay.

In [ ]:
# NOTE: embedding all 15 videos inline (embed=True) produced a >150MB
# notebook that IDE notebook renderers (VS Code/Cursor webviews) fail to
# open (renders blank) -- video files are already written to disk by the
# render cell above, so just print clickable paths instead of re-embedding.
for seed in SEEDS:
  print(f'=== seed {seed} ===')
  for path in video_paths[seed]:
    print(f'  {path}')


## Find successful rollouts

NUM_VIDEOS_PER_CHECKPOINT random draws above will often show zero successes purely by chance even when the checkpoint's true success rate is nonzero -- see the close-subtask sibling notebook's note on this for the binomial-draw reasoning (e.g. at a measured ~20% success rate, P(0 successes in 5 draws) is ~33%). This cell instead keeps sampling fresh rollouts per checkpoint until it finds TARGET_SUCCESSES_PER_CHECKPOINT episodes where the drawer actually opens, or gives up after MAX_ATTEMPTS_PER_CHECKPOINT tries -- saving only the successful ones. Uses stochastic=False (deterministic policy.mode()) to match the render-videos cell above; if that turns out to never succeed for this env (as measured for the close-subtask sibling's checkpoints), switch to stochastic=True here.

In [ ]:
TARGET_SUCCESSES_PER_CHECKPOINT = 3
MAX_ATTEMPTS_PER_CHECKPOINT = 60

success_video_paths = {seed: [] for seed in SEEDS}
for seed in SEEDS:
  label, ckpt_path = checkpoint_paths[seed]
  print(f'\n=== seed={seed}  checkpoint={label}  searching for successes ===')
  ckpt = ppo_learner.load_checkpoint(ckpt_path)
  policy_params = ckpt['policy_params']

  obs_norm_state = ckpt.get('obs_normalizer_state')
  obs_normalizer = (
      ppo_learner.ObsNormalizer.from_state_dict(obs_norm_state)
      if obs_norm_state is not None else None)

  out_dir = os.path.join(OUTPUT_ROOT, f'seed_{seed}_success_search')
  os.makedirs(out_dir, exist_ok=True)

  n_success = 0
  attempt = -1
  for attempt in range(MAX_ATTEMPTS_PER_CHECKPOINT):
    frames, stats = ppo_video_utils.rollout_and_render(
        policy_params=policy_params,
        gym_env=gym_env,
        networks=networks,
        render_fn=render_fn,
        max_steps=env_max_steps,
        stochastic=False,  # deterministic mode() never succeeds -- see note above
        seed=10_000 + attempt,
        obs_normalizer=obs_normalizer,
        success_fn=_drawer_open_success_fn,
    )
    tag = 'SUCCESS' if stats['success'] else 'fail'
    print(f'  attempt {attempt:2d}: length={stats["length"]:4d}  '
          f'drawer_open={stats["success"]}  [{tag}]')
    if stats['success']:
      out_path = os.path.join(out_dir, f'success_{n_success}_attempt{attempt}.mp4')
      ppo_video_utils.write_video(frames, out_path, fps=FPS)
      success_video_paths[seed].append(out_path)
      print(f'    -> saved {out_path}')
      n_success += 1
      if n_success >= TARGET_SUCCESSES_PER_CHECKPOINT:
        break

  hit_rate = 100.0 * n_success / (attempt + 1)
  print(f'  seed={seed}: found {n_success} success(es) in {attempt + 1} attempts ({hit_rate:.1f}%)')

print('\nDone searching. Successful rollouts:')
for seed in SEEDS:
  print(f'=== seed {seed} ===')
  for path in success_video_paths[seed]:
    print(f'  {path}')

## Cleanup

Run this last, once you're done with both rendering sections above (they share the same `gym_env`).

In [ ]:
gym_env.close()
print('closed gym_env')